# 배정 비율이 어긋나면 왜 결과를 버려야 하는가

**질문**: 50:50으로 배정했는데 49:51이 나왔다. 문제인가?

**결론**: 표본 크기에 따라 다르다. 같은 48:52라도 1,000명이면
우연이고 10,000명이면 경보다. 그리고 SRM이 확인되면 보정하지 않고
실험 결과 전체를 폐기한다.

다른 함정들과 대응 방식이 다르다. 엿보기와 다중검정은 기준을
조정해 대응하지만, SRM은 조정할 방법이 없다.

In [1]:
import pandas as pd

from abtest_lab.diagnostics import check_srm

## 1. 같은 비율, 다른 판정

배정 비율을 48:52로 고정하고 전체 인원만 늘려가며 판정을 본다.
카이제곱 검정으로 "관측된 인원이 의도한 비율과 얼마나 다른가"를 잰다.

In [2]:
rows = []
for n in [1000, 2000, 5000, 10000, 20000]:
    counts = [int(n * 0.48), int(n * 0.52)]
    r = check_srm(counts)
    rows.append({
        "전체 인원": n,
        "배정": f"{counts[0]}:{counts[1]}",
        "p값": f"{r['p_value']:.2e}",
        "SRM": r["srm"],
    })

pd.DataFrame(rows)

,전체 인원,배정,p값,SRM
0,1000,480:520,2.06e-01,False
1,2000,960:1040,7.36e-02,False
2,5000,2400:2600,4.68e-03,False
3,10000,4800:5200,6.33e-05,True
4,20000,9600:10400,1.54e-08,True


전부 48:52다. 그런데 5,000명과 10,000명 사이에서 판정이 갈린다.

표본이 크면 우연의 흔들림이 줄어들기 때문이다. 1,000명을 무작위로
나누면 480:520 정도는 흔하지만, 20,000명에서 9,600:10,400이
나오려면 우연만으로는 설명하기 어렵다.

5,000명 행의 p값은 0.0047이다. 일반 검정 기준 0.05였다면 경보였겠지만
SRM 기준 0.001에서는 통과한다. 기준을 어디에 두느냐로 판정이 갈리는
구간이며, 다음 절에서 그 이유를 본다.

**실무 함의**: 트래픽이 큰 서비스일수록 SRM이 자주 잡힌다. 도구가
예민해서가 아니라 작은 어긋남도 우연으로 설명되지 않기 때문이며,
실제로 문제가 있는 경우가 대부분이다.

## 2. 왜 기준이 0.001인가

지금까지 다룬 검정은 0.05를 썼다. SRM은 0.001을 쓴다.
49:51이라는 애매한 사례로 기준을 바꿔가며 본다.

In [3]:
rows = []
for alpha in [0.05, 0.01, 0.001]:
    r = check_srm([4900, 5100], alpha=alpha)
    rows.append({
        "기준(alpha)": alpha,
        "p값": round(r["p_value"], 4),
        "판정": "SRM" if r["srm"] else "정상",
    })

pd.DataFrame(rows)

,기준(alpha),p값,판정
0,0.050,0.0455,SRM
1,0.010,0.0455,정상
2,0.001,0.0455,정상


p값은 0.0455로 동일하다. 달라진 것은 기준뿐인데 판정이 뒤집힌다.

실험을 수백 개 운영하는 조직에서 0.05를 쓰면 정상 실험의 5%가
경보를 울린다. 경보가 잦으면 아무도 믿지 않게 되고, 진짜 문제가
생겼을 때도 무시된다. SRM은 결과 폐기라는 큰 대가를 요구하므로
확실할 때만 울려야 한다.

대가는 있다. 0.001을 쓰면 어느 정도의 어긋남은 놓친다.
경계선 사례는 기록만 해두고 넘어가되, 같은 실험 그룹에서
반복되면 조사하는 식으로 다룬다.

## 3. 의도한 불균등 배분

신기능을 소수에게만 노출하는 실험에서는 배정이 균등하지 않다.
같은 900:100 데이터를 두 가지 기준으로 판정해본다.

In [4]:
r1 = check_srm([900, 100], expected_ratio=[0.9, 0.1])
r2 = check_srm([900, 100])

pd.DataFrame([
    {"의도한 비율": "90:10", "p값": f"{r1['p_value']:.2e}", "SRM": r1["srm"]},
    {"의도한 비율": "50:50 (기본값)", "p값": f"{r2['p_value']:.2e}", "SRM": r2["srm"]},
])

,의도한 비율,p값,SRM
0,90:10,1.00e+00,False
1,50:50 (기본값),3.34e-141,True


같은 데이터인데 판정이 정반대다. 도구가 틀린 것이 아니라
**의도를 알려주지 않았기 때문**이다.

SRM 체크는 "실제 배정이 의도와 같은가"를 묻는 것이지 "균등한가"를
묻는 것이 아니다. 실험 설계 문서에 배정 비율을 명시해두어야 하는
이유이기도 하다.

## 4. 보정이 아니라 폐기인 이유

엿보기나 다중검정은 보정으로 대응할 수 있다. 몇 번 봤는지 세어서
기준을 조정하면 된다. SRM은 다르다.

배정 인원이 어긋났다는 것은 **사라진 사용자가 있다**는 뜻이다.
그리고 그들은 무작위로 사라지지 않았을 가능성이 높다.

- A안 페이지가 느려서 로딩 중 이탈 → 참을성 없는 사용자가 빠짐
- 특정 브라우저에서 B안 코드 오류 → 그 브라우저 사용자가 빠짐
- 봇 트래픽이 한쪽에 몰림

어떤 사람이 어떤 이유로 빠졌는지 모르므로 보정할 방법이 없다.
남은 데이터로 계산한 전환율은 원래 모집단의 전환율이 아니다.

그래서 SRM 체크는 다른 분석보다 **먼저** 한다.
통과하지 못한 실험의 전환율 비교는 의미가 없다.

## 5. SRM 발생 시 확인 순서

1. **배정 로직** — 랜덤 시드, 해시 함수가 균등한가
2. **분석 쿼리의 필터 조건** — 한쪽만 걸러내고 있지 않은가
3. **기록 유실** — 한쪽 페이지가 느려 이벤트가 덜 남지 않는가
4. **리다이렉트** — 한쪽만 페이지 이동이 있어 이탈이 생기지 않는가

2번이 의외로 흔하다. 실험 자체는 정상인데 봇 제외 조건 같은 것이
한쪽에 불리하게 걸린 경우다. 코드가 아니라 분석 쿼리를 봐야 한다.